In [ ]:
# %% Deep learning - Section 23.209
#    Code challenge 36: linear GAN with FMNIST
#
#    1) Start from code from video 23.208
#    2) Use FMNIST data
#    3) Train on 3 image classes:
#       a) Trouser, sneakers, pullover
#       b) Trouser, sneakers, sandals
#    4) Use DataLoaders and Batches (train on 5k batches)
#    5) Smooth the plotting of the losses

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2
import torchvision
import torchvision.transforms as T
import torch.nn.utils         as utils
import random

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from matplotlib.gridspec              import GridSpec
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [33]:
# %% FMNIST data

# Define transform (ToTensor brings to [0,1], and Normalise to [-1,1])
transform = T.Compose([ T.ToTensor(),
                        T.Normalize(.5,.5) ])

# Import data and apply the transform
dataset = torchvision.datasets.FashionMNIST(root='./data',train=True,download=True,transform=transform)

# Select classes
print(dataset.classes)
keep_classes = [ 'Trouser','Sneaker', 'Pullover'  ]
keep_classes = [ 'Trouser','Sneaker', 'Sandal'  ]

class_indices = [dataset.classes.index(c) for c in keep_classes]
mask          = torch.isin(dataset.targets, torch.tensor(class_indices))
images2use    = torch.where(mask)[0]

# Convert into DataLoader objects
batch_size  = 100
sampler     = torch.utils.data.SubsetRandomSampler(images2use)
data_loader = DataLoader(dataset,sampler=sampler,batch_size=batch_size,drop_last=True)


In [7]:
# %% Discriminator class

class DiscriminatorNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(28*28,256)
        self.fc2 = nn.Linear(  256,256)
        self.out = nn.Linear(  256,1  )

    def forward(self,x):

        x = F.leaky_relu( self.fc1(x))
        x = F.leaky_relu( self.fc2(x))
        x = torch.sigmoid(self.out(x))

        return x


In [8]:
# %% Generator class

class GeneratorNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear( 64,256  )
        self.fc2 = nn.Linear(256,256  )
        self.out = nn.Linear(256,28*28)

    def forward(self,x):

        x = self.fc1(x)
        x = F.leaky_relu(x)
        x = self.fc2(x)
        x = F.leaky_relu(x)
        x = torch.tanh(self.out(x))

        return x


In [ ]:
# %% Test classes on random data

# Discriminator
d_net = DiscriminatorNet()
y     = d_net(torch.randn(10,28*28))
print(y.shape)
print(y), print()

# Generator
g_net = GeneratorNet()
y     = g_net(torch.randn(10,64))
print(y.shape)
print(y[0,:].detach().squeeze().view(28,28).shape)


In [34]:
# %% Set up the GAN ensemble

# Loss funtion is the same for both training phases
loss_fun = nn.BCELoss()

# Model instances
d_net = DiscriminatorNet().to(device)
g_net = GeneratorNet().to(device)

# Optimizer (same but two instances for the two models; GANs usually have a
# small lr and train slowly)
d_optimizer = torch.optim.Adam(d_net.parameters(),lr=0.0003)
g_optimizer = torch.optim.Adam(g_net.parameters(),lr=0.0003)


In [ ]:
# %% Train the GAN

# Takes ~20 mins on GPU with 50K batches
num_epochs = int(50000/len(data_loader))

# Preallocate
losses    = np.zeros((num_epochs*len(data_loader),2))
decisions = np.zeros((num_epochs*len(data_loader),2))
loss_i    = 0

# Loop
for epoch_i in range(num_epochs):

    for data,_ in data_loader:

        # Minibatches of real and fake images
        data = data.view(batch_size,-1).to(device)

        # Label for real and fake iamges
        real_labels = torch.ones(batch_size,1).to(device)
        fake_labels = torch.zeros(batch_size,1).to(device)

        # Train the discriminator

        # Forward propagationa and loss for real images (all labels are 1)
        pred_real   = d_net(data)
        d_loss_real = loss_fun(pred_real,real_labels)

        # Forward propagation and loss for fake images (all lablels are 0)
        fake_imgs   = g_net(torch.randn(batch_size,64).to(device))
        pred_fake   = d_net(fake_imgs)
        d_loss_fake = loss_fun(pred_fake,fake_labels)

        # Build combined loss (note as you can scale the loss to make the model more
        # or less sensitive to FP or FN)
        d_loss = 1*d_loss_real + 1*d_loss_fake

        losses[loss_i,0]    = d_loss.item()
        decisions[loss_i,0] = torch.mean((pred_real>.5).float()).detach()

        # Backpropagation of discriminator
        d_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()

        # Train the generator

        # Create fake images and compute loss
        fake_imgs  = g_net(torch.randn(batch_size,64).to(device))
        pred_fake  = d_net(fake_imgs)

        # Get loss and discrimination (pass real labels so that the model minimise
        # the loss between pred_fake and real_labels)
        g_loss = loss_fun(pred_fake,real_labels)
        losses[loss_i,1]    = g_loss.item()
        decisions[loss_i,1] = torch.mean((pred_fake>.5).float()).detach()

        # Backpropagation of generator
        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

        # Loss counter
        loss_i += 1

    # Status message
    msg = f'Finished epoch {epoch_i+1}/{num_epochs}'
    sys.stdout.write('\r' + msg)


In [17]:
# %% Functions for 1D smoothing filter

# Improved for edge effects - adaptive window
def smooth_adaptive(x,k):
    smoothed = np.zeros_like(x)
    half_k   = k // 2

    for i in range(len(x)):
        start       = max(0, i-half_k)
        end         = min(len(x), i+half_k + 1)
        smoothed[i] = np.mean(x[start:end])

    return smoothed


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,3,figsize=(2*phi*6,6))

ax[0].plot(smooth_adaptive(losses[:,0],500),alpha=0.75)
ax[0].plot(smooth_adaptive(losses[:,1],500),alpha=0.75)
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].set_title('Model losses')
ax[0].legend(['Discrimator','Generator'])
# ax[0].set_xlim([4000,5000])

ax[1].plot(losses[::5,0],losses[::5,1],'.',color='tab:red',alpha=.1)
ax[1].set_xlabel('Discriminator loss')
ax[1].set_ylabel('Generator loss')
ax[1].set_title('Disc. Loss vs. Gen. Loss')

ax[2].plot(smooth_adaptive(decisions[:,0],500),alpha=0.75)
ax[2].plot(smooth_adaptive(decisions[:,1],500),alpha=0.75)
ax[2].set_xlabel('Epochs')
ax[2].set_ylabel('Probablity ("real")')
ax[2].set_title('Discriminator output')
ax[2].legend(['Real','Fake'])

plt.savefig('figure16_code_challenge_36.png')
plt.show()
files.download('figure16_code_challenge_36.png')


In [ ]:
# %% Plotting

# Generate the images with the generator network
g_net.eval()
fake_data = g_net(torch.randn(12,64).to(device)).cpu()

# Plot
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(3,4,figsize=(phi*6,6))

for i,ax in enumerate(axs.flatten()):
    ax.imshow(fake_data[i,:,].detach().view(28,28),cmap='gray')
    ax.axis('off')

plt.suptitle(keep_classes,y=.95,fontweight='bold')

plt.savefig('figure17_code_challenge_36.png')
plt.show()
files.download('figure17_code_challenge_36.png')
